# Text preprocessing

Raw text is a sequence of characters. Before any model can use it, we need to decide what the units are, which units matter, and how to represent them as numbers. This notebook covers tokenisation, stopword removal, stemming, lemmatisation, bag-of-words and TF-IDF.

Order: intuition, pipeline overview, from-scratch tokeniser, NLTK, spaCy, bag of words, TF-IDF, what breaks.

In [ ]:
import math
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import spacy
from nltk.corpus import gutenberg, stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (7, 3)

## 1. Intuition

A model trained on text sees tokens, not meaning. Two sentences that share no words look completely different even if they say the same thing. Preprocessing reduces vocabulary size, normalises surface forms, and removes tokens that carry no information, which makes the remaining signal stronger.

In [ ]:
raw = "Emma Woodhouse, handsome, clever, and rich, had lived nearly twenty-one years in the world."
print("raw tokens (split):  ", raw.split())
print("count:", len(raw.split()))

Splitting on spaces keeps punctuation attached to words. `"clever,"` and `"clever"` become different tokens. The comma carries no meaning. It just inflates the vocabulary.

## 2. Pipeline overview

A standard preprocessing pipeline runs these steps in order. Each step is a function that maps text to text or tokens to tokens.

In [ ]:
def preprocess_pipeline(text: str, stemmer: PorterStemmer | None = None) -> list[str]:
    """Lowercase → tokenise → remove stopwords → stem or lemmatise."""
    stop = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop]
    if stemmer:
        return [stemmer.stem(t) for t in tokens]
    return [lemmatizer.lemmatize(t) for t in tokens]


print("pipeline output:", preprocess_pipeline(raw))

## 3. From scratch: tokeniser

A whitespace split is the simplest tokeniser. It fails on punctuation, contractions and hyphenated words. A regex tokeniser handles these cases explicitly.

In [ ]:
def whitespace_tokenise(text: str) -> list[str]:
    return text.split()


example = "It's twenty-one years, isn't it? Yes!"
print("whitespace:", whitespace_tokenise(example))

In [ ]:
def regex_tokenise(text: str) -> list[str]:
    # Keep contractions and hyphenated words as single tokens; drop pure punctuation.
    return re.findall(r"\b[\w]+(?:[-'][\w]+)*\b", text.lower())


print("regex:     ", regex_tokenise(example))

The regex tokeniser preserves `"it's"` and `"twenty-one"` as single units. Whether that is right depends on the vocabulary you want. Many pipelines split contractions (`"it's"` → `["it", "is"]`) to reduce sparsity.

## 4. NLTK

`word_tokenize` uses a Punkt tokeniser trained on English. It handles most punctuation edge cases and splits contractions. The Porter stemmer applies a fixed set of suffix rules. The WordNet lemmatiser looks up the base form in a dictionary.

In [ ]:
# Short excerpt from Jane Austen's Emma (NLTK Gutenberg corpus).
emma_raw = gutenberg.raw("austen-emma.txt")
excerpt = emma_raw[:300]
print(excerpt)

In [ ]:
stop_en = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

tokens_raw = word_tokenize(excerpt.lower())
tokens_clean = [t for t in tokens_raw if t.isalpha() and t not in stop_en]

print("raw token count:    ", len(tokens_raw))
print("after stopword removal:", len(tokens_clean))
print("sample tokens:", tokens_clean[:12])

Stopword removal cuts the token count roughly in half. The remaining tokens carry most of the content.

In [ ]:
# Compare stemming vs lemmatisation on a selection of words.
sample_words = ["running", "studies", "better", "wolves", "caring", "easily", "fairly"]
print(f"{'word':<12} {'stem':<12} {'lemma':<12}")
for w in sample_words:
    print(f"{w:<12} {stemmer.stem(w):<12} {lemmatizer.lemmatize(w):<12}")

The stem is a chopped string, fast but sometimes not a real word (`"studi"`). The lemma is the dictionary base form. Use lemmatisation when the tokens need to be human-readable; use stemming when speed matters and vocabulary size is the goal.

## 5. spaCy

spaCy runs a full NLP pipeline: tokenisation, POS tagging, dependency parsing, NER. `en_core_web_sm` is the small English model. Each token carries the original text, lemma, POS tag, and a stop flag.

In [ ]:
nlp = spacy.load("en_core_web_sm")
doc = nlp(
    "Emma Woodhouse had lived nearly twenty-one years in the world with very little to distress her."
)

print(f"{'text':<14} {'lemma':<14} {'pos':<8} {'stop'}")
print("-" * 46)
for token in doc:
    print(f"{token.text:<14} {token.lemma_:<14} {token.pos_:<8} {token.is_stop}")

spaCy lemmatises `"had"` to `"have"` and correctly tags it as a verb. The WordNet lemmatiser would return `"had"` without a POS hint. The POS information is useful for disambiguation: `"better"` as a verb (`"to better something"`) has a different lemma than `"better"` as an adjective.

## 6. Bag of words

A bag-of-words (BoW) representation counts how many times each vocabulary word appears in a document and ignores order. The result is a vector of counts, one entry per vocabulary word.

In [ ]:
docs = [
    "the cat sat on the mat",
    "the dog sat on the floor",
    "the cat chased the dog",
    "a mat is on the floor",
]

# From scratch: count each word per document.
vocab = sorted({w for doc in docs for w in doc.split()})
bow_scratch = [[doc.split().count(w) for w in vocab] for doc in docs]

print("vocabulary:", vocab)
print("matrix (rows=docs, cols=vocab words):")
for i, row in enumerate(bow_scratch):
    print(f"  doc{i}: {row}")

In [ ]:
cv = CountVectorizer()
X_bow = cv.fit_transform(docs)  # scipy sparse matrix
print("sklearn vocab:", cv.get_feature_names_out())
print(
    "shape:",
    X_bow.shape,
    "  stored elements:",
    X_bow.nnz,
    "  total cells:",
    X_bow.shape[0] * X_bow.shape[1],
)
print("density:", round(X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1]), 2))

With a vocabulary of thousands of words and thousands of documents, most cells are zero. A sparse matrix stores only the non-zero entries. For a 10,000-document corpus with a 20,000-word vocabulary, the dense matrix would need 1.6 GB; the sparse version typically needs under 10 MB.

## 7. TF-IDF

Raw counts favour long documents and common words. TF-IDF corrects both. **Term frequency** (TF) is the fraction of a document that the term occupies. **Inverse document frequency** (IDF) down-weights words that appear in many documents:

$$\text{TF}(t, d) = \frac{\text{count}(t, d)}{|d|} \qquad \text{IDF}(t) = \log\frac{N}{|\{d : t \in d\}|}$$

The final score is their product. A word that appears often in one document but rarely across the corpus scores high.

In [ ]:
def tfidf_from_scratch(documents: list[str]) -> tuple[np.ndarray, list[str]]:
    """Return (tfidf matrix, vocab list). Rows = docs, cols = vocab."""
    tokenised = [doc.lower().split() for doc in documents]
    vocab = sorted({w for doc in tokenised for w in doc})
    w2i = {w: i for i, w in enumerate(vocab)}
    n = len(documents)
    matrix = np.zeros((n, len(vocab)))
    for d_idx, tokens in enumerate(tokenised):
        counts = Counter(tokens)
        total = len(tokens)
        for word, cnt in counts.items():
            tf = cnt / total
            df = sum(1 for doc in tokenised if word in doc)
            idf = math.log(n / df)
            matrix[d_idx, w2i[word]] = tf * idf
    return matrix, vocab


matrix, vocab_tfidf = tfidf_from_scratch(docs)
print("vocab:", vocab_tfidf)
print("TF-IDF matrix:")
print(np.round(matrix, 3))

In [ ]:
# Words appearing in every document get IDF = log(1) = 0. Confirm that.
universal_words = [w for w in vocab_tfidf if all(w in doc.lower() for doc in docs)]
print("words in all docs (zero IDF):", universal_words)

In [ ]:
# sklearn TfidfVectorizer uses a smoothed IDF to avoid division by zero.
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(docs).toarray()
feature_names = tfidf.get_feature_names_out()
print("sklearn TF-IDF (rounded):")
print(np.round(X_tfidf, 3))
print("features:", feature_names)

In [ ]:
# Which words distinguish each document most?
for i, row in enumerate(X_tfidf):
    top_idx = row.argsort()[::-1][:3]
    top = [(feature_names[j], round(row[j], 3)) for j in top_idx if row[j] > 0]
    print(f"doc{i}: {docs[i]!r}")
    print(f"  top terms: {top}")

## 8. What breaks

- **Out-of-vocabulary tokens.** A vocabulary built on training data has no entry for unseen words at test time. Subword tokenisation (BPE, WordPiece) handles this by splitting into known pieces.
- **Morphologically rich languages.** Finnish or Turkish have thousands of valid surface forms per root. Stemming and lemmatisation rules are language-specific and require separate models.
- **Compound words.** German `"Donaudampfschifffahrtsgesellschaft"` is one word with many concepts. English `"machine learning"` is two words with one concept.
- **Negation.** `"not good"` and `"good"` share the word `"good"`. A BoW or TF-IDF representation cannot distinguish them without n-gram features or separate negation handling.
- **Word order.** BoW and TF-IDF discard sequence entirely. `"dog bites man"` and `"man bites dog"` are identical vectors. The next notebooks address this.